# Scrape Sofascore Match Data

This notebook downloads match-level data from Sofascore for a given league.

**Change `LEAGUE` in the next cell** to switch between leagues:
- `'cg'` → Montenegrin 1. CFL (tournament 154)
- `'srb'` → Serbian Mozzart Bet Superliga (tournament 210)

Steps:
- Fetch / load match IDs from `matches.csv`
- For each match, scrape: lineups, positions, shots, momentum, team stats, heatmaps
- Download player profile photos

### Configure paths, imports, and the Selenium driver used for every request.

In [11]:
# ─── League Selection ─────────────────────────────────────────
LEAGUE = 'srb'  # 'cg' = Montenegro 1. CFL | 'srb' = Serbian SuperLiga

# ─── Imports ──────────────────────────────────────────────────
from pathlib import Path
import json
import sys
import pandas as pd
from time import sleep
from typing import Iterable
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# ─── Load league config ──────────────────────────────────────
ROOT = Path('..')
sys.path.insert(0, str(ROOT / 'src'))
from config import LEAGUES

league_cfg = LEAGUES[LEAGUE]
LEAGUE_ID  = league_cfg['tournament_id']
SEASON_ID  = league_cfg['season_id']
print(f"League: {league_cfg['name']} ({league_cfg['country']})")
print(f"Tournament ID: {LEAGUE_ID}, Season ID: {SEASON_ID}")

# ─── Path Configuration ──────────────────────────────────────
DATA_RAW = ROOT / 'data' / 'raw' / LEAGUE
OUT_DIR  = DATA_RAW
RAW_DIR  = OUT_DIR / 'raw_by_match'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

PLAYER_PHOTOS_DIR = ROOT / 'data' / 'processed' / LEAGUE / 'player_photos'
PLAYER_PHOTOS_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://api.sofascore.com/api/v1"

# ─── Selenium WebDriver setup ────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
print("Successfully initialized Selenium WebDriver.")

League: Mozzart Bet Superliga (Serbia)
Tournament ID: 210, Season ID: 76909
Successfully initialized Selenium WebDriver.


### Load match IDs plus basic metadata so downstream helpers know team and slug info.

In [12]:
# Load match IDs from the existing matches.csv file (if it exists)
MATCH_METADATA: dict[int, dict] = {}

matches_csv_path = OUT_DIR / "matches.csv"
if matches_csv_path.exists():
    matches_df = pd.read_csv(matches_csv_path)
    match_ids = sorted(matches_df["id"].dropna().astype(int).unique().tolist())
    print(f"Loaded {len(match_ids)} match IDs from existing matches.csv")

    # Build a quick lookup so we always know each match's slug and team names/IDs
    for _, row in matches_df.iterrows():
        mid = int(row["id"])
        MATCH_METADATA[mid] = {
            "slug": row.get("slug"),
            "home_team_id": row.get("homeTeam.id"),
            "home_team_name": row.get("homeTeam.name"),
            "away_team_id": row.get("awayTeam.id"),
            "away_team_name": row.get("awayTeam.name"),
        }
else:
    match_ids = []
    print(f"No matches.csv found — fresh league. Run the incremental update cell to fetch matches.")

Loaded 229 match IDs from existing matches.csv


### Define the per-endpoint fetcher and helpers that reshape player stats into tab-friendly tables.

In [13]:
# Scrape per-match data using Selenium

def get_json_from_url(url: str):
    """Use Selenium to fetch the content of a URL and parse it as JSON."""
    if not driver:
        raise RuntimeError("Selenium driver is not initialized.")
    driver.get(url)
    pre_tag = driver.find_element(By.TAG_NAME, "pre")
    return json.loads(pre_tag.text)

def _safe_pct(value, total):
    """Return a percentage (0-100) or None if it cannot be computed."""
    if value is None or total in (None, 0):
        return None
    return (float(value) / float(total)) * 100

def _format_number(value, precision=None):
    if value is None:
        return ""
    if isinstance(value, (int, float)):
        if precision is None:
            if isinstance(value, float) and not value.is_integer():
                precision = 2
            else:
                precision = 0
        formatted = f"{value:.{precision}f}"
        if "." in formatted:
            formatted = formatted.rstrip("0").rstrip(".")
        return formatted
    return str(value)

PLAYER_STATS_CATEGORY_CONFIG = {
    "General": [
        {"name": "Goals", "value_key": "goals"},
        {"name": "Assists", "value_key": "goalAssist"},
        {"name": "Tackles (won)", "value_key": "totalTackle", "secondary_key": "wonTackle"},
        {"name": "Accurate passes", "value_key": "accuratePass", "denominator_key": "totalPass", "percentage_key": "pass_accuracy_pct"},
        {"name": "Duels (won)", "value_key": "total_duels", "secondary_key": "duelWon"},
        {"name": "Ground duels (won)", "value_key": "ground_duels_total", "secondary_key": "ground_duels_won"},
        {"name": "Aerial duels (won)", "value_key": "aerial_duels_total", "secondary_key": "aerialWon"},
        {"name": "Minutes played", "value_key": "minutesPlayed"},
    ],
    "Attacking": [
        {"name": "Goals", "value_key": "goals"},
        {"name": "Shots (on target)", "value_key": "totalShots", "secondary_key": "onTargetScoringAttempt"},
        {"name": "Shots off target", "value_key": "shotOffTarget"},
        {"name": "Shots blocked", "value_key": "blockedScoringAttempt"},
        {"name": "Expected goals", "value_key": "expectedGoals", "value_precision": 2},
        {"name": "Assists", "value_key": "goalAssist"},
        {"name": "Expected assists", "value_key": "expectedAssists", "value_precision": 2},
        {"name": "Key passes", "value_key": "keyPass"},
        {"name": "Dribbles (successful)", "value_key": "totalContest", "secondary_key": "wonContest", "percentage_key": "dribble_success_pct"},
        {"name": "Touches", "value_key": "touches"},
        {"name": "Was fouled", "value_key": "wasFouled"},
    ],
    "Defending": [
        {"name": "Tackles (won)", "value_key": "totalTackle", "secondary_key": "wonTackle"},
        {"name": "Interceptions", "value_key": "interceptionWon"},
        {"name": "Ball recoveries", "value_key": "ballRecovery"},
        {"name": "Clearances", "value_key": "totalClearance"},
        {"name": "Blocks", "value_key": "outfielderBlock"},
        {"name": "Dispossessed", "value_key": "dispossessed"},
        {"name": "Errors leading to shot", "value_key": "errorLeadToAShot"},
        {"name": "Own goals", "value_key": "ownGoals"},
    ],
    "Passing": [
        {"name": "Accurate passes", "value_key": "accuratePass", "denominator_key": "totalPass", "percentage_key": "pass_accuracy_pct"},
        {"name": "Accurate long balls", "value_key": "accurateLongBalls", "denominator_key": "totalLongBalls", "percentage_key": "long_ball_accuracy_pct"},
        {"name": "Accurate crosses", "value_key": "accurateCross", "denominator_key": "totalCross", "percentage_key": "cross_accuracy_pct"},
        {"name": "Passes own half", "value_key": "accurateOwnHalfPasses", "denominator_key": "totalOwnHalfPasses"},
        {"name": "Passes opposition half", "value_key": "accurateOppositionHalfPasses", "denominator_key": "totalOppositionHalfPasses"},
        {"name": "Key passes", "value_key": "keyPass"},
        {"name": "Expected assists", "value_key": "expectedAssists", "value_precision": 2},
    ],
    "Duels": [
        {"name": "Duels (won)", "value_key": "total_duels", "secondary_key": "duelWon"},
        {"name": "Ground duels (won)", "value_key": "ground_duels_total", "secondary_key": "ground_duels_won"},
        {"name": "Aerial duels (won)", "value_key": "aerial_duels_total", "secondary_key": "aerialWon"},
        {"name": "Was fouled", "value_key": "wasFouled"},
        {"name": "Fouls committed", "value_key": "fouls"},
        {"name": "Dispossessed", "value_key": "dispossessed"},
        {"name": "Possession lost", "value_key": "possessionLostCtrl"},
    ],
    "Goalkeeping": [
        {"name": "Saves", "value_key": "saves"},
        {"name": "Saves (inside box)", "value_key": "savedShotsFromInsideTheBox"},
        {"name": "Keeper sweepers (accurate)", "value_key": "totalKeeperSweeper", "secondary_key": "accurateKeeperSweeper", "percentage_key": "keeper_sweeper_pct"},
        {"name": "Good high claims", "value_key": "goodHighClaim"},
        {"name": "Crosses not claimed", "value_key": "crossNotClaimed"},
        {"name": "Minutes played", "value_key": "minutesPlayed"},
    ],
}

def enrich_player_statistics(raw_stats):
    stats = dict(raw_stats or {})
    duel_won = stats.get("duelWon") or 0
    duel_lost = stats.get("duelLost") or 0
    aerial_won = stats.get("aerialWon") or 0
    aerial_lost = stats.get("aerialLost") or 0
    total_duels = duel_won + duel_lost
    aerial_total = aerial_won + aerial_lost
    stats["total_duels"] = total_duels
    stats["aerial_duels_total"] = aerial_total
    stats["ground_duels_total"] = max(total_duels - aerial_total, 0)
    stats["ground_duels_won"] = max(duel_won - aerial_won, 0)
    stats["ground_duels_lost"] = max(duel_lost - aerial_lost, 0)
    stats["pass_accuracy_pct"] = _safe_pct(stats.get("accuratePass"), stats.get("totalPass"))
    stats["long_ball_accuracy_pct"] = _safe_pct(stats.get("accurateLongBalls"), stats.get("totalLongBalls"))
    stats["cross_accuracy_pct"] = _safe_pct(stats.get("accurateCross"), stats.get("totalCross"))
    stats["dribble_success_pct"] = _safe_pct(stats.get("wonContest"), stats.get("totalContest"))
    stats["keeper_sweeper_pct"] = _safe_pct(stats.get("accurateKeeperSweeper"), stats.get("totalKeeperSweeper"))
    return stats

def build_player_rows(match_id, lineups_data):
    rows = []
    meta = MATCH_METADATA.get(match_id, {})
    for side in ("home", "away"):
        side_data = lineups_data.get(side, {})
        team_name = meta.get(f"{side}_team_name") or side
        team_id = meta.get(f"{side}_team_id")
        for entry in side_data.get("players", []):
            player_info = entry.get("player", {})
            stats = enrich_player_statistics(entry.get("statistics", {}))
            row = {
                "match_id": match_id,
                "team_side": side,
                "team_id": entry.get("teamId") or team_id,
                "team_name": team_name,
                "player_id": player_info.get("id"),
                "player_name": player_info.get("name") or player_info.get("shortName"),
                "player_slug": player_info.get("slug"),
                "player_position": entry.get("position") or player_info.get("position"),
                "shirt_number": entry.get("shirtNumber") or player_info.get("jerseyNumber"),
                "is_substitute": bool(entry.get("substitute")),
            }
            row.update(stats)
            rows.append(row)
    return rows

def _build_metric_display(value, denominator, secondary, percentage, metric_def):
    if value is None and denominator is None and secondary is None:
        return ""
    if denominator not in (None, 0):
        primary = _format_number(value, metric_def.get("value_precision"))
        denom = _format_number(denominator, metric_def.get("denominator_precision"))
        base = f"{primary}/{denom}"
        if percentage is not None:
            pct_prec = metric_def.get("percentage_precision", 0)
            base = f"{base} ({percentage:.{pct_prec}f}%)"
        return base.strip()
    primary = _format_number(value, metric_def.get("value_precision"))
    if secondary is not None:
        secondary_str = _format_number(secondary, metric_def.get("secondary_precision"))
        return f"{primary} ({secondary_str})".strip()
    if percentage is not None:
        pct_prec = metric_def.get("percentage_precision", 0)
        return f"{primary} ({percentage:.{pct_prec}f}%)".strip()
    return primary

def expand_player_stat_categories(match_id, player_rows):
    records = []
    for row in player_rows:
        for category, metrics in PLAYER_STATS_CATEGORY_CONFIG.items():
            for metric in metrics:
                value = row.get(metric["value_key"])
                denominator = row.get(metric.get("denominator_key"))
                secondary = row.get(metric.get("secondary_key"))
                percentage = row.get(metric.get("percentage_key"))
                if percentage is None and denominator not in (None, 0):
                    percentage = _safe_pct(value, denominator)
                display = _build_metric_display(value, denominator, secondary, percentage, metric)
                records.append({
                    "match_id": match_id,
                    "team_id": row.get("team_id"),
                    "team_name": row.get("team_name"),
                    "team_side": row.get("team_side"),
                    "player_id": row.get("player_id"),
                    "player_name": row.get("player_name"),
                    "player_position": row.get("player_position"),
                    "metric_key": metric["value_key"],
                    "category": category,
                    "metric": metric["name"],
                    "value": value,
                    "secondary": secondary,
                    "denominator": denominator,
                    "percentage": percentage,
                    "display": display,
                })
    return records

def generate_player_stats_tables(match_id, lineups_data, match_dir):
    player_rows = build_player_rows(match_id, lineups_data)
    if not player_rows:
        return
    base_df = pd.DataFrame(player_rows)
    base_df.to_csv(match_dir / "player_base_stats.csv", index=False)
    metrics_df = pd.DataFrame(expand_player_stat_categories(match_id, player_rows))
    metrics_df.to_csv(match_dir / "player_stats_tabs.csv", index=False)

def scrape_match_details(match_id: int, base_dir: Path) -> None:
    """
    Scrape all available data for a single Sofascore match.
    
    Endpoints scraped:
    - lineups: Player statistics (minutes, passes, tackles, goals, assists, etc.)
    - avg_positions: Player position tracking data
    - shotmap: Shot-level data with xG
    - graph: Match momentum over time
    - statistics: Team-level statistics
    - heatmaps: Player heatmap data
    """
    match_dir = base_dir / str(int(match_id))
    match_dir.mkdir(parents=True, exist_ok=True)

    endpoints = {
        "lineups": f"{BASE_URL}/event/{match_id}/lineups",
        "avg_positions": f"{BASE_URL}/event/{match_id}/average-positions",
        "shotmap": f"{BASE_URL}/event/{match_id}/shotmap",
        "graph": f"{BASE_URL}/event/{match_id}/graph",
        "statistics": f"{BASE_URL}/event/{match_id}/statistics",
        "heatmaps": f"{BASE_URL}/event/{match_id}/player-heatmaps",
    }

    for name, url in endpoints.items():
        data = get_json_from_url(url)

        if name == "avg_positions":
            if isinstance(data, dict) and 'home' in data and 'away' in data:
                home_df = pd.DataFrame(data['home'])
                home_df['team'] = 'home'
                away_df = pd.DataFrame(data['away'])
                away_df['team'] = 'away'
                combined_df = pd.concat([home_df, away_df], ignore_index=True)
                combined_df.to_csv(match_dir / "avg_positions.csv", index=False)
            elif isinstance(data, list):
                pd.DataFrame(data).to_csv(match_dir / "avg_positions.csv", index=False)
            else:
                print(f"  - Skipping avg_positions for match {match_id} due to unexpected format.")
        elif name == "graph":
            pd.DataFrame(data.get('graphPoints', [])).to_csv(match_dir / "match_momentum.csv", index=False)
        elif name == "shotmap":
            pd.DataFrame(data.get('shotmap', [])).to_csv(match_dir / "match_shots.csv", index=False)
        elif name == "lineups":
            with (match_dir / "lineups.json").open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False)
            generate_player_stats_tables(match_id, data, match_dir)
        else:
            with (match_dir / f"{name}.json").open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False)

### Provide a simple wrapper that iterates match IDs and retries each scrape sequentially.

In [14]:
def scrape_all_matches(match_ids: Iterable[int], output_dir: Path, pause_seconds: float = 1.0) -> None:
    """Iterate over match IDs, scraping each one sequentially."""
    ids = list(match_ids or [])
    if not ids:
        print("❌ No match IDs provided; nothing to scrape.")
        return

    output_dir.mkdir(parents=True, exist_ok=True)
    total = len(ids)
    print(f"ℹ️ Scraping {total} matches...")

    for index, match_id in enumerate(ids, start=1):
        print(f"[{index}/{total}] Scraping match {match_id}")
        scrape_match_details(int(match_id), output_dir)
        if pause_seconds:
            sleep(pause_seconds)

    print("\n✅ All matches scraped successfully.")

### Kick off the end-to-end scrape using the currently loaded match list.

In [15]:
# Run the full scrape

ids_to_scrape = match_ids
print(f"🚀 Starting scrape for {len(ids_to_scrape)} matches into {RAW_DIR}")
print(f"📊 Will collect: lineups (player stats), positions, shots, momentum, team stats")
scrape_all_matches(ids_to_scrape, RAW_DIR, pause_seconds=1.0)
print("\n✅ Scraping complete!")
print(f"📁 Data saved to: {RAW_DIR}")
print("\n📋 Each match folder now contains:")
print("   - lineups.json (player stats with minutes played, passes, tackles, etc.)")
print("   - avg_positions.csv")
print("   - match_shots.csv")
print("   - match_momentum.csv")
print("   - statistics.json (team-level)")
print("   - heatmaps.json")

🚀 Starting scrape for 229 matches into ..\data\raw\srb\raw_by_match
📊 Will collect: lineups (player stats), positions, shots, momentum, team stats
ℹ️ Scraping 229 matches...
[1/229] Scraping match 14015386
[2/229] Scraping match 14015387
[3/229] Scraping match 14015388
[4/229] Scraping match 14015389
[5/229] Scraping match 14015390
[6/229] Scraping match 14015391
[7/229] Scraping match 14015392
  - Skipping avg_positions for match 14015392 due to unexpected format.
[8/229] Scraping match 14015393
[9/229] Scraping match 14015394
  - Skipping avg_positions for match 14015394 due to unexpected format.
[10/229] Scraping match 14015395
[11/229] Scraping match 14015396
[12/229] Scraping match 14015397
[13/229] Scraping match 14015398
[14/229] Scraping match 14015399
[15/229] Scraping match 14015400
[16/229] Scraping match 14015401
[17/229] Scraping match 14015402
[18/229] Scraping match 14015403
  - Skipping avg_positions for match 14015403 due to unexpected format.
[19/229] Scraping match 1

## Scrape Player Photos

Download profile photos for all players. Players without available photos on Sofascore are skipped.

In [16]:
# Player photo scraping functions

from tqdm.auto import tqdm

def extract_player_ids_from_lineups(raw_dir: Path) -> set:
    """Extract all unique player IDs from lineups.json files."""
    player_ids = set()
    for match_dir in raw_dir.iterdir():
        if match_dir.is_dir():
            lineups_file = match_dir / "lineups.json"
            if lineups_file.exists():
                with lineups_file.open("r", encoding="utf-8") as f:
                    data = json.load(f)
                for team in ("home", "away"):
                    for player_data in data.get(team, {}).get("players", []):
                        player_id = (player_data.get("player") or {}).get("id")
                        if player_id:
                            player_ids.add(player_id)
    return player_ids

def download_player_photo(player_id: int, output_dir: Path) -> tuple[bool, str]:
    """Download a player photo from Sofascore. Returns (success, message)."""
    url = f"https://img.sofascore.com/api/v1/player/{player_id}/image"
    output_path = output_dir / f"{player_id}.png"
    
    if output_path.exists():
        return True, "Already exists"

    driver.get(url)
    sleep(0.8)
    
    # Check for error response - skip if no photo available
    if '"error"' in driver.page_source:
        return False, "No photo available"
    
    # Find and save the image
    img_elements = driver.find_elements(By.TAG_NAME, "img")
    if img_elements:
        img_data = img_elements[0].screenshot_as_png
        if img_data and len(img_data) > 1000:
            with output_path.open("wb") as f:
                f.write(img_data)
            return True, "Success"
    
    return False, "No image found"

def scrape_player_photos(player_ids: set, output_dir: Path) -> None:
    """Download photos for all players, skipping those without photos."""
    output_dir.mkdir(parents=True, exist_ok=True)
    successful, failed = 0, 0
    errors: dict[str, int] = {}

    for player_id in tqdm(player_ids, desc="Downloading photos"):
        success, msg = download_player_photo(player_id, output_dir)
        if success:
            successful += 1
        else:
            failed += 1
            errors[msg] = errors.get(msg, 0) + 1
        sleep(0.5)

    print(f"\n✅ Downloaded: {successful} | ❌ Skipped: {failed}")
    if errors:
        for error, count in errors.items():
            print(f"   - {error}: {count}")

In [17]:
# Scrape player photos

# Get all player IDs from match data
player_ids = extract_player_ids_from_lineups(RAW_DIR)
print(f"Found {len(player_ids)} unique players")

# Download photos (skips existing, skips players without photos)
scrape_player_photos(player_ids, PLAYER_PHOTOS_DIR)

Found 600 unique players



✅ Downloaded: 563 | ❌ Skipped: 37
   - No photo available: 37


## Scrape Team Logos

Download team logos from SofaScore for all teams in the league.

In [20]:
# Team logo scraping functions

TEAM_LOGOS_DIR = ROOT / 'data' / 'processed' / LEAGUE / 'team_logos'
TEAM_LOGOS_DIR.mkdir(parents=True, exist_ok=True)

def extract_team_ids_from_lineups(raw_dir: Path) -> set:
    """Extract all unique team IDs from lineups.json files."""
    team_ids = set()
    for match_dir in raw_dir.iterdir():
        if match_dir.is_dir():
            lineups_file = match_dir / 'lineups.json'
            if lineups_file.exists():
                with lineups_file.open('r', encoding='utf-8') as f:
                    data = json.load(f)
                for side in ('home', 'away'):
                    team_data = data.get(side, {})
                    # Team ID is on the first player entry
                    players = team_data.get('players', [])
                    if players:
                        tid = players[0].get('teamId')
                        if tid:
                            team_ids.add(tid)
    return team_ids

def download_team_logo(team_id: int, output_dir: Path) -> tuple[bool, str]:
    """Download a team logo from SofaScore. Returns (success, message)."""
    url = f'https://img.sofascore.com/api/v1/team/{team_id}/image'
    output_path = output_dir / f'{team_id}.png'

    if output_path.exists():
        return True, 'Already exists'

    driver.get(url)
    sleep(0.8)

    if '"error"' in driver.page_source:
        return False, 'No logo available'

    img_elements = driver.find_elements(By.TAG_NAME, 'img')
    if img_elements:
        img_data = img_elements[0].screenshot_as_png
        if img_data and len(img_data) > 500:
            with output_path.open('wb') as f:
                f.write(img_data)
            return True, 'Success'

    return False, 'No image found'

def scrape_team_logos(team_ids: set, output_dir: Path) -> None:
    """Download logos for all teams."""
    output_dir.mkdir(parents=True, exist_ok=True)
    successful, failed = 0, 0

    for team_id in sorted(team_ids):
        success, msg = download_team_logo(team_id, output_dir)
        if success:
            successful += 1
            print(f'  {team_id}: {msg}')
        else:
            failed += 1
            print(f'  {team_id}: {msg}')
        sleep(0.5)

    print(f'Downloaded: {successful} | Skipped: {failed}')

# Get all team IDs and download logos
team_ids = extract_team_ids_from_lineups(RAW_DIR)
print(f'Found {len(team_ids)} teams')
scrape_team_logos(team_ids, TEAM_LOGOS_DIR)


Found 20 teams
  3115: Success
  5144: Success
  5148: Success
  5149: Success
  5150: Success
  5152: Success
  5153: Success
  7674: Success
  7771: Success
  25736: Success
  25737: Success
  36967: Success
  43981: Success
  48614: Success
  111127: Success
  229348: Success
  241802: Success
  263360: Success
  282203: Success
  308176: Success
Downloaded: 20 | Skipped: 0


## Incremental Update

Fetch only new matches since the last scrape. This cell:
1. Fetches the latest match list from Sofascore API
2. Compares with existing matches in `matches.csv`
3. Appends new matches to the CSV
4. Scrapes details for only the new matches

In [18]:
# Incremental scraping - Fetch and scrape only new matches
# LEAGUE_ID and SEASON_ID are set from config in the first cell

def fetch_all_matches_from_api(league_id: int, season_id: int) -> pd.DataFrame:
    """Fetch all matches for a season from Sofascore API."""
    all_matches = []
    page = 0
    
    while True:
        url = f"{BASE_URL}/unique-tournament/{league_id}/season/{season_id}/events/last/{page}"
        print(f"  Fetching page {page}...")
        data = get_json_from_url(url)
        
        events = data.get("events", [])
        if not events:
            break
            
        all_matches.extend(events)
        
        if not data.get("hasNextPage", False):
            break
        page += 1
        sleep(0.5)
    
    # Flatten nested JSON into DataFrame
    df = pd.json_normalize(all_matches)
    return df

def get_new_match_ids(api_df: pd.DataFrame, existing_csv_path: Path) -> tuple[pd.DataFrame, list[int]]:
    """Compare API matches with existing CSV and return new matches."""
    if existing_csv_path.exists():
        existing_df = pd.read_csv(existing_csv_path)
        existing_ids = set(existing_df["id"].dropna().astype(int).tolist())
    else:
        existing_df = pd.DataFrame()
        existing_ids = set()
    
    api_ids = set(api_df["id"].dropna().astype(int).tolist())
    new_ids = api_ids - existing_ids
    
    # Filter API dataframe to only new matches
    new_matches_df = api_df[api_df["id"].isin(new_ids)].copy()
    
    return new_matches_df, sorted(new_ids)

def run_incremental_update():
    """Main function to fetch new matches and scrape their details."""
    print(f"🔍 Fetching matches from Sofascore API (tournament {LEAGUE_ID}, season {SEASON_ID})...")
    api_df = fetch_all_matches_from_api(LEAGUE_ID, SEASON_ID)
    print(f"   Found {len(api_df)} total matches in API")
    
    # Compare with existing
    matches_csv_path = OUT_DIR / "matches.csv"
    new_matches_df, new_ids = get_new_match_ids(api_df, matches_csv_path)
    
    if not new_ids:
        print("\n✅ No new matches found. Everything is up to date!")
        return
    
    print(f"\n📊 Found {len(new_ids)} new matches to scrape:")
    for mid in new_ids:
        print(f"   - Match ID: {mid}")
    
    # Append new matches to CSV
    print(f"\n📝 Appending new matches to {matches_csv_path}...")
    if matches_csv_path.exists():
        existing_df = pd.read_csv(matches_csv_path)
        # Ensure columns match
        for col in existing_df.columns:
            if col not in new_matches_df.columns:
                new_matches_df[col] = None
        for col in new_matches_df.columns:
            if col not in existing_df.columns:
                existing_df[col] = None
        combined_df = pd.concat([existing_df, new_matches_df[existing_df.columns]], ignore_index=True)
    else:
        combined_df = new_matches_df
    
    combined_df.to_csv(matches_csv_path, index=False)
    print(f"   ✅ matches.csv now contains {len(combined_df)} matches")
    
    # Update MATCH_METADATA for new matches
    for _, row in new_matches_df.iterrows():
        mid = int(row["id"])
        MATCH_METADATA[mid] = {
            "slug": row.get("slug"),
            "home_team_id": row.get("homeTeam.id"),
            "home_team_name": row.get("homeTeam.name"),
            "away_team_id": row.get("awayTeam.id"),
            "away_team_name": row.get("awayTeam.name"),
        }
    
    # Scrape details for new matches only
    print(f"\n🚀 Scraping details for {len(new_ids)} new matches...")
    scrape_all_matches(new_ids, RAW_DIR, pause_seconds=1.0)
    
    print("\n✅ Incremental update complete!")
    print(f"   - New matches added: {len(new_ids)}")
    print(f"   - Total matches in CSV: {len(combined_df)}")
    print("\n💡 Remember to re-run 02_data_cleaning.ipynb to update processed data!")

# Run the incremental update
run_incremental_update()

🔍 Fetching matches from Sofascore API (tournament 210, season 76909)...
  Fetching page 0...
  Fetching page 1...
  Fetching page 2...
  Fetching page 3...
  Fetching page 4...
  Fetching page 5...
  Fetching page 6...
  Fetching page 7...
   Found 229 total matches in API

✅ No new matches found. Everything is up to date!
